#### **Explainable Post-Disaster Grid Observability Recovery Using Human-Oversight Agentic LLMs**

##### Authors
**Biswas Rudra Jyoti Arka**<sup>1</sup>, **Sadman Sakib**<sup>2</sup>, **Md. Zahidul Islam**<sup>1,*</sup>, **Shamsun Nahar Edib**<sup>2</sup>

<sup>1</sup> School of Electrical, Computer, and Biomedical Engineering, Southern Illinois University Carbondale, IL 62901, USA
<sup>2</sup> Department of Electrical and Computer Engineering, Montana State University, Bozeman, MT 59717, USA

<sup>*</sup> **Corresponding author:** [mdzahidul.islam@siu.edu](mailto:mdzahidul.islam@siu.edu)



Example Tool Definition (corresponds to Section III-A1)

In [ ]:
import json

PMU_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "propose_restoration",
            "description": "Propose PMU restorations for all zones; return restoration_plan without applying. Always pass budgets from compute_zone_scores_and_budgets. You MUST include rationale (short text) explaining why these budgets and this restoration step are appropriate.",
            "parameters": {
                "type": "object",
                "properties": {
                    "zone_nets": {"type": "object", "description": "Zone pandapower networks by zone id."},
                    "zone_pmu_buses": {"type": "object", "description": "PMU candidate buses by zone."},
                    "region_buses": {"type": "object", "description": "Bus lists by zone."},
                    "zone_states": {"type": "object", "description": "Current PMU ON/OFF states by zone."},
                    "n_zones": {"type": "integer", "description": "Number of zones."},
                    "horizon": {"type": "integer", "description": "Lookahead horizon.", "default": 3},
                    "budgets": {
                        "description": "Per-zone budgets from compute_zone_scores_and_budgets (order 0..n_zones-1), or one integer for all zones.",
                        "oneOf": [
                            {"type": "integer"},
                            {"type": "array", "items": {"type": "integer", "minimum": 0}}
                        ]
                    },
                    "rationale": {
                        "type": "string",
                        "description": "Required: Reason for this restoration step and the selection of budgets for each zones based on the scores computed in compute_zone_scores_and_budgets.",
                    },
                },
                "required": ["zone_nets", "zone_pmu_buses", "region_buses", "zone_states", "n_zones", "budgets", "rationale"],
            },
        },
    },
    

]

Tool Execution (corresponds to Section III-A2)

In [ ]:
def execute_tool(name, arguments, context=None):
    """Execute one PMU tool call by name and return JSON text for the LLM."""
    args = arguments if isinstance(arguments, dict) else json.loads(arguments or "{}")
    if context is None:
        context = {}

    try:
        
        if name == "propose_restoration":
            zb = args.get("budgets", context.get("zone_budgets"))
            if zb is not None and hasattr(zb, "tolist"):
                zb = zb.tolist()
            rationale = args.get("rationale", "")
            context["last_propose_rationale"] = rationale
            restoration_plan = propose_restoration(
                zone_nets=args.get("zone_nets", context.get("zone_nets")),
                zone_pmu_buses=args.get("zone_pmu_buses", context.get("zone_pmu_buses")),
                region_buses=args.get("region_buses", context.get("region_buses")),
                zone_states=args.get("zone_states", context.get("zone_states")),
                n_zones=args.get("n_zones", context.get("n_zones")),
                horizon=args.get("horizon", 3),
                budgets=zb if zb is not None else 0,
            )
            context["restoration_plan"] = restoration_plan
            return json.dumps({
                "rationale": rationale,
                "restoration_plan": {
                    str(k): {
                        "activated": v["activated"],
                        "n_obs": v["n_obs"],
                        "budget": v["budget"],
                        "new_status": v["new_status"].tolist(),
                        "obs": v["obs"].tolist(),
                    }
                    for k, v in restoration_plan.items()
                }
            })

        return json.dumps({"error": f"Unknown tool: {name}"})

    except Exception as e:
        return json.dumps({"error": str(e)})

System Prompt and User Prompt (corresponds to Section III-B1)

In [ ]:
system = f"""
    You are an operator restoring PMU observability.

    You must follow the workflow in this exact order:
    1. Call partition_network(net_choice={net_choice}, n_zones={n_zones}).
    2. Call assign_random_zone_weights(n_zones={n_zones}, rng_seed={weights_rng_seed}).
    3. Call generate_failure_scenario(..., n_zones={n_zones}, failure_rates={fr_list}, rng_seed={failure_rng_seed}).
    4. While the zones are NOT fully observable, do:
       a) Call compute_zone_scores_and_budgets(..., budget={budget}, n_zones={n_zones}, alpha={alpha}, beta={beta}) to get per-zone budgets.
       b) Call propose_restoration(..., n_zones={n_zones}, budgets=zone_budgets, rationale=<non-empty reason>) and give an explanation for the proposed restoration plan and the selected budgets.
       c) Call apply_restoration(..., n_zones={n_zones}, restoration_plan=latest).
       d) Call check_total_observability(..., n_zones={n_zones}).
    5. Call evaluate_tieline_observability(..., n_zones={n_zones}).
    6. You must provide a rationale for the restoration plan and the selected budgets explaining with numbers, why these budgets and this restoration step are appropriate.
    6. DO NOT call evaluate_tieline_observability until the zones are fully observable.
    
    Final answer must report restored PMUs in order and final observability status.
    You must follow the workflow untill the grid is fully observable or the max iterations is reached.
    You must use tool outputs and these fixed run arguments; do not invent values.
    """

prompt = (
        "Execute the PMU restoration workflow using the available tool inputs and prior tool outputs. "
        "Use this exact order: partition_network, assign_random_zone_weights, generate_failure_scenario, "
        "then iterate compute_zone_scores_and_budgets -> propose_restoration (include budgets and rationale) "
        "-> apply_restoration -> check_total_observability until the grid is fully observable, "
        "then run evaluate_tieline_observability. "
        "Final response must include  restored PMUs in order and final observability status."
    )